In [81]:
from nba_api.stats.endpoints import LeagueDashLineups
from nba_api.stats.endpoints import TeamPlayerOnOffDetails
from nba_api.stats.endpoints import LeagueDashPlayerStats
from nba_api.stats.endpoints import LeagueHustleStatsPlayer
from nba_api.stats.endpoints import LeagueDashPtStats

from nba_api.stats.static import teams

from sklearn.model_selection import TimeSeriesSplit, cross_val_score, cross_validate, GridSearchCV
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, root_mean_squared_error
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import SimpleImputer, IterativeImputer
from sklearn.compose import make_column_transformer

import os
import numpy as np
import time
import random
import pandas as pd
from requests.exceptions import ConnectionError, Timeout
from functools import reduce


# OREB Model

#### Target:
- On/Off OReb% change

#### Features to try:
- Individual OReb%
- Interior gravity? - not enough data to train on, only goes back to 2025-26

Tracking (goes back to 2013-14):
- Contested OReb%
- OReb chance
- deferred OReb chances?
- adjusted oreb %?
- AVG oreb dist

Box outs (goes back to 2017-18):
- % offensive boxouts
- % team/player rebound when boxout

Usage (goes back to 1996-97):
- % of teams oreb

##### Feature engineering to try:
- calculate league average, calculate features relative to league average

#### Other things to try:
- weighting by k years ago, D^k for each value. (from RAPM article)
- "We use a weighting scheme in which data from k years ago is weighted
with weighting param D^k, for various values of D."

In [19]:
general_adv_df = LeagueDashPlayerStats(
                season="2025-26",
                season_type_all_star="Regular Season",
                measure_type_detailed_defense="Advanced",
                per_mode_detailed="Per100Possessions",
                timeout=60
            ).get_data_frames()[0]

In [20]:
general_adv_df.columns

Index(['PLAYER_ID', 'PLAYER_NAME', 'NICKNAME', 'TEAM_ID', 'TEAM_ABBREVIATION',
       'AGE', 'GP', 'W', 'L', 'W_PCT', 'MIN', 'E_OFF_RATING', 'OFF_RATING',
       'sp_work_OFF_RATING', 'E_DEF_RATING', 'DEF_RATING',
       'sp_work_DEF_RATING', 'E_NET_RATING', 'NET_RATING',
       'sp_work_NET_RATING', 'AST_PCT', 'AST_TO', 'AST_RATIO', 'OREB_PCT',
       'DREB_PCT', 'REB_PCT', 'TM_TOV_PCT', 'E_TOV_PCT', 'EFG_PCT', 'TS_PCT',
       'USG_PCT', 'E_USG_PCT', 'E_PACE', 'PACE', 'PACE_PER40', 'sp_work_PACE',
       'PIE', 'POSS', 'FGM', 'FGA', 'FGM_PG', 'FGA_PG', 'FG_PCT', 'GP_RANK',
       'W_RANK', 'L_RANK', 'W_PCT_RANK', 'MIN_RANK', 'E_OFF_RATING_RANK',
       'OFF_RATING_RANK', 'sp_work_OFF_RATING_RANK', 'E_DEF_RATING_RANK',
       'DEF_RATING_RANK', 'sp_work_DEF_RATING_RANK', 'E_NET_RATING_RANK',
       'NET_RATING_RANK', 'sp_work_NET_RATING_RANK', 'AST_PCT_RANK',
       'AST_TO_RANK', 'AST_RATIO_RANK', 'OREB_PCT_RANK', 'DREB_PCT_RANK',
       'REB_PCT_RANK', 'TM_TOV_PCT_RANK', 'E_TOV_PCT_

In [ ]:
adv_oreb_cols = ["PLAYER_ID", "OREB_PCT"]

In [153]:
# Rebound Tracking data
tracking_reb_df = LeagueDashPtStats(
                season="2025-26",
                season_type_all_star="Regular Season",
                pt_measure_type="Rebounding",
                per_mode_simple="Totals",
                player_or_team="Player",
                timeout=60
            ).get_data_frames()[0]

ReadTimeout: HTTPSConnectionPool(host='stats.nba.com', port=443): Read timed out. (read timeout=60)

In [11]:
tracking_reb_df.columns

Index(['PLAYER_ID', 'PLAYER_NAME', 'TEAM_ID', 'TEAM_ABBREVIATION', 'GP', 'W',
       'L', 'MIN', 'OREB', 'OREB_CONTEST', 'OREB_UNCONTEST',
       'OREB_CONTEST_PCT', 'OREB_CHANCES', 'OREB_CHANCE_PCT',
       'OREB_CHANCE_DEFER', 'OREB_CHANCE_PCT_ADJ', 'AVG_OREB_DIST', 'DREB',
       'DREB_CONTEST', 'DREB_UNCONTEST', 'DREB_CONTEST_PCT', 'DREB_CHANCES',
       'DREB_CHANCE_PCT', 'DREB_CHANCE_DEFER', 'DREB_CHANCE_PCT_ADJ',
       'AVG_DREB_DIST', 'REB', 'REB_CONTEST', 'REB_UNCONTEST',
       'REB_CONTEST_PCT', 'REB_CHANCES', 'REB_CHANCE_PCT', 'REB_CHANCE_DEFER',
       'REB_CHANCE_PCT_ADJ', 'AVG_REB_DIST'],
      dtype='object')

In [13]:
tracking_reb_df["OREB_DEFER_PCT"] = tracking_reb_df["OREB_CHANCE_DEFER"] / tracking_reb_df["OREB_CHANCES"]
tracking_reb_df["DREB_DEFER_PCT"] = tracking_reb_df["DREB_CHANCE_DEFER"] / tracking_reb_df["DREB_CHANCES"]

In [ ]:
tracking_oreb_cols = [
    "PLAYER_ID",
    "OREB_CONTEST_PCT",
    "OREB_CHANCE_PCT",
    "AVG_OREB_DIST",
    "OREB_DEFER_PCT",
]

In [62]:
hustle_df = LeagueHustleStatsPlayer(
                season="2025-26",
                season_type_all_star="Regular Season",
                per_mode_time="Totals",
                timeout=60
            ).get_data_frames()[0]

In [207]:
hustle_df.columns

Index(['PLAYER_ID', 'PLAYER_NAME', 'TEAM_ID', 'TEAM_ABBREVIATION', 'AGE', 'G',
       'MIN', 'CONTESTED_SHOTS', 'CONTESTED_SHOTS_2PT', 'CONTESTED_SHOTS_3PT',
       'DEFLECTIONS', 'CHARGES_DRAWN', 'SCREEN_ASSISTS', 'SCREEN_AST_PTS',
       'OFF_LOOSE_BALLS_RECOVERED', 'DEF_LOOSE_BALLS_RECOVERED',
       'LOOSE_BALLS_RECOVERED', 'PCT_LOOSE_BALLS_RECOVERED_OFF',
       'PCT_LOOSE_BALLS_RECOVERED_DEF', 'OFF_BOXOUTS', 'DEF_BOXOUTS',
       'BOX_OUT_PLAYER_TEAM_REBS', 'BOX_OUT_PLAYER_REBS', 'BOX_OUTS',
       'PCT_BOX_OUTS_OFF', 'PCT_BOX_OUTS_DEF', 'PCT_BOX_OUTS_TEAM_REB',
       'PCT_BOX_OUTS_REB'],
      dtype='object')

In [ ]:
hustle_oreb_cols = [
    "PLAYER_ID",
    "PCT_BOX_OUTS_OFF",
    # Not offensive specific:
    "PCT_BOX_OUTS_TEAM_REB",
    "PCT_BOX_OUTS_REB"
]

In [29]:
usage_df = LeagueDashPlayerStats(
                season="2025-26",
                season_type_all_star="Regular Season",
                measure_type_detailed_defense="Usage",
                per_mode_detailed="Per100Possessions",
                timeout=60
            ).get_data_frames()[0]

In [31]:
usage_df.columns

Index(['PLAYER_ID', 'PLAYER_NAME', 'NICKNAME', 'TEAM_ID', 'TEAM_ABBREVIATION',
       'AGE', 'GP', 'W', 'L', 'W_PCT', 'MIN', 'USG_PCT', 'PCT_FGM', 'PCT_FGA',
       'PCT_FG3M', 'PCT_FG3A', 'PCT_FTM', 'PCT_FTA', 'PCT_OREB', 'PCT_DREB',
       'PCT_REB', 'PCT_AST', 'PCT_TOV', 'PCT_STL', 'PCT_BLK', 'PCT_BLKA',
       'PCT_PF', 'PCT_PFD', 'PCT_PTS', 'GP_RANK', 'W_RANK', 'L_RANK',
       'W_PCT_RANK', 'MIN_RANK', 'USG_PCT_RANK', 'PCT_FGM_RANK',
       'PCT_FGA_RANK', 'PCT_FG3M_RANK', 'PCT_FG3A_RANK', 'PCT_FTM_RANK',
       'PCT_FTA_RANK', 'PCT_OREB_RANK', 'PCT_DREB_RANK', 'PCT_REB_RANK',
       'PCT_AST_RANK', 'PCT_TOV_RANK', 'PCT_STL_RANK', 'PCT_BLK_RANK',
       'PCT_BLKA_RANK', 'PCT_PF_RANK', 'PCT_PFD_RANK', 'PCT_PTS_RANK',
       'TEAM_COUNT'],
      dtype='object')

In [ ]:
usage_oreb_cols = [
    "PLAYER_ID",
    "PCT_OREB" # % of team's offensive rebounds
    ]

In [33]:
usage_df.loc[usage_df['PLAYER_NAME'] == "CJ McCollum"]

,PLAYER_ID,PLAYER_NAME,NICKNAME,TEAM_ID,TEAM_ABBREVIATION,AGE,GP,W,L,W_PCT,...,PCT_REB_RANK,PCT_AST_RANK,PCT_TOV_RANK,PCT_STL_RANK,PCT_BLK_RANK,PCT_BLKA_RANK,PCT_PF_RANK,PCT_PFD_RANK,PCT_PTS_RANK,TEAM_COUNT
70,203468,CJ McCollum,CJ,1610612737,ATL,34.0,76,37,39,0.487,...,499,153,409,386,327,270,85,219,63,2


### Actual code starts here

In [151]:
usage_args = {
    "season": "2020-21",
    "season_type_all_star": "Regular Season",
    "measure_type_detailed_defense": "Usage",
    "per_mode_detailed": "Per100Possessions",
    # "timeout": 60
}

In [52]:
usage_args.values()

dict_values(['2025-26', 'Regular Season', 'Usage', 'Per100Possessions'])

In [5]:
def build_cache_str(cache_dir, func_str, arg_values):
    cache_str = cache_dir + "/" + func_str
    
    for arg in arg_values:
        cache_str = cache_str + "_" + str(arg).replace(" ", "_")
    cache_str = cache_str + ".csv"

    return cache_str

In [10]:
build_cache_str("cache", "LeagueDashPlayerStats", usage_args.values())

'cache/LeagueDashPlayerStats_2025-26_Regular_Season_Usage_Per100Possessions.csv'

In [11]:
def cached_api_call(endpoint_func, endpoint_func_str, endpoint_args, dataframe_idx=0, cache_dir="cache"):

    os.makedirs(cache_dir, exist_ok=True)

    path = build_cache_str(cache_dir, endpoint_func_str, endpoint_args.values())

    # load cache
    if os.path.exists(path):
        print("Loading cached:", path)
        return pd.read_csv(path)


    retries = 8

    for attempt in range(retries):

        try:

            print(
                f"Downloading {path}, attempt {attempt+1}/{retries}"
            )

            df = endpoint_func(**endpoint_args, timeout=60).get_data_frames()[dataframe_idx]

            df.to_csv(path, index=False)

            print("Saved", path)

            time.sleep(random.uniform(2,4))

            return df


        except (ConnectionError, Timeout) as e:

            wait = 10 * (attempt + 1)

            print(
                f"Connection failed. Waiting {wait}s..."
            )

            time.sleep(wait)


    raise Exception(
        f"Could not download lineup data for {path}"
    )

    

In [206]:
test_df = cached_api_call(LeagueDashPlayerStats, "LeagueDashPlayerStats", usage_args)

KeyboardInterrupt: 

In [24]:
# Get target data from lineups
def get_four_factors_targets(season):

    rows = []

    for team in teams.get_teams():

        print(team["full_name"])

        target_args = {
            "season": season,
            "season_type_all_star": "Regular Season",
            "measure_type_detailed_defense": "Four Factors",
            "per_mode_detailed": "Per100Possessions",
            "team_id": team["id"],
        }

        data = cached_api_call(TeamPlayerOnOffDetails, "TeamPlayerOnOffDetails", target_args, dataframe_idx=1)

        data = data[[
            'VS_PLAYER_ID',
            'MIN',
            'EFG_PCT', 'FTA_RATE', 'TM_TOV_PCT', 'OREB_PCT',
            'OPP_EFG_PCT', 'OPP_FTA_RATE', 'OPP_TOV_PCT', 'OPP_OREB_PCT',
        ]]

        data = data.rename(columns={
            "VS_PLAYER_ID": "PLAYER_ID", 
            "EFG_PCT": "TM_EFG_PCT", 
            "FTA_RATE": "TM_FTA_RATE", 
            "OREB_PCT": "TM_OREB_PCT"})

        rows.append(data)


        time.sleep(1)


    return pd.concat(rows, ignore_index=True)

In [161]:
test_target_df = get_four_factors_targets("2020-21")

Atlanta Hawks
Connection failed. Waiting 10s...


KeyboardInterrupt: 

In [49]:
def merge_traded_players(df):
    rows = []

    weighted_cols = ['TM_EFG_PCT', 'TM_FTA_RATE', 'TM_TOV_PCT', 'TM_OREB_PCT',
       'OPP_EFG_PCT', 'OPP_FTA_RATE', 'OPP_TOV_PCT', 'OPP_OREB_PCT']
    
    for (player_id), group in df.groupby("PLAYER_ID"):
        w = group["MIN"]

        row = {
            "PLAYER_ID": player_id,
            "MIN": group["MIN"].sum(),
        }

        # print(row['PLAYER_ID'])

        for col in weighted_cols:
            row[col] = np.average(group[col], weights=w)

        

        rows.append(row)

    return pd.DataFrame(rows)

In [50]:
merge_traded_players(test_target_df)

,PLAYER_ID,MIN,TM_EFG_PCT,TM_FTA_RATE,TM_TOV_PCT,TM_OREB_PCT,OPP_EFG_PCT,OPP_FTA_RATE,OPP_TOV_PCT,OPP_OREB_PCT
0,2544,1989.0,0.572,0.308,0.144,0.291,0.550,0.229,0.147,0.304
1,101108,228.0,0.504,0.183,0.180,0.286,0.571,0.290,0.160,0.323
2,200768,118.0,0.452,0.348,0.122,0.294,0.530,0.278,0.141,0.367
3,201142,2840.0,0.550,0.274,0.157,0.382,0.537,0.254,0.132,0.292
4,201143,969.0,0.554,0.218,0.140,0.274,0.562,0.228,0.161,0.336
...,...,...,...,...,...,...,...,...,...,...
577,1643133,49.0,0.555,0.240,0.140,0.316,0.577,0.297,0.168,0.333
578,1643141,32.0,0.490,0.500,0.164,0.267,0.509,0.291,0.162,0.324
579,1643158,169.0,0.513,0.259,0.170,0.340,0.572,0.284,0.164,0.305
580,1643253,217.0,0.491,0.162,0.133,0.232,0.606,0.316,0.134,0.367


In [212]:
seasons = [
    # Don't have these cached yet:
    # "2017-18",
    # "2018-19",
    # "2019-20",
    # "2020-21",
    "2021-22",
    "2022-23",
    "2023-24",
    "2024-25",
    "2025-26"
]

In [214]:
targets_dict = {}

for season in seasons:

    print("Starting", season)
    
    targets_df = get_four_factors_targets(season)

    targets_dict[season] = merge_traded_players(targets_df)
    time.sleep(2)

Starting 2021-22
Atlanta Hawks
Loading cached: cache/TeamPlayerOnOffDetails_2021-22_Regular_Season_Four_Factors_Per100Possessions_1610612737.csv
Boston Celtics
Loading cached: cache/TeamPlayerOnOffDetails_2021-22_Regular_Season_Four_Factors_Per100Possessions_1610612738.csv
Cleveland Cavaliers
Loading cached: cache/TeamPlayerOnOffDetails_2021-22_Regular_Season_Four_Factors_Per100Possessions_1610612739.csv
New Orleans Pelicans
Loading cached: cache/TeamPlayerOnOffDetails_2021-22_Regular_Season_Four_Factors_Per100Possessions_1610612740.csv
Chicago Bulls
Loading cached: cache/TeamPlayerOnOffDetails_2021-22_Regular_Season_Four_Factors_Per100Possessions_1610612741.csv
Dallas Mavericks
Loading cached: cache/TeamPlayerOnOffDetails_2021-22_Regular_Season_Four_Factors_Per100Possessions_1610612742.csv
Denver Nuggets
Loading cached: cache/TeamPlayerOnOffDetails_2021-22_Regular_Season_Four_Factors_Per100Possessions_1610612743.csv
Golden State Warriors
Loading cached: cache/TeamPlayerOnOffDetails_20

In [53]:
targets_dict["2025-26"]

,PLAYER_ID,MIN,TM_EFG_PCT,TM_FTA_RATE,TM_TOV_PCT,TM_OREB_PCT,OPP_EFG_PCT,OPP_FTA_RATE,OPP_TOV_PCT,OPP_OREB_PCT
0,2544,1989.0,0.572,0.308,0.144,0.291,0.550,0.229,0.147,0.304
1,101108,228.0,0.504,0.183,0.180,0.286,0.571,0.290,0.160,0.323
2,200768,118.0,0.452,0.348,0.122,0.294,0.530,0.278,0.141,0.367
3,201142,2840.0,0.550,0.274,0.157,0.382,0.537,0.254,0.132,0.292
4,201143,969.0,0.554,0.218,0.140,0.274,0.562,0.228,0.161,0.336
...,...,...,...,...,...,...,...,...,...,...
577,1643133,49.0,0.555,0.240,0.140,0.316,0.577,0.297,0.168,0.333
578,1643141,32.0,0.490,0.500,0.164,0.267,0.509,0.291,0.162,0.324
579,1643158,169.0,0.513,0.259,0.170,0.340,0.572,0.284,0.164,0.305
580,1643253,217.0,0.491,0.162,0.133,0.232,0.606,0.316,0.134,0.367


In [17]:
test_df[test_df['PLAYER_NAME'] == 'Jaren Jackson Jr.']

,PLAYER_ID,PLAYER_NAME,NICKNAME,TEAM_ID,TEAM_ABBREVIATION,AGE,GP,W,L,W_PCT,...,PCT_REB_RANK,PCT_AST_RANK,PCT_TOV_RANK,PCT_STL_RANK,PCT_BLK_RANK,PCT_BLKA_RANK,PCT_PF_RANK,PCT_PFD_RANK,PCT_PTS_RANK,TEAM_COUNT
251,1628991,Jaren Jackson Jr.,Jaren,1610612762,UTA,26.0,48,19,29,0.396,...,262,435,450,193,70,527,510,54,47,2


In [ ]:
oreb_df_dict = {}

for season in seasons:

    print("Starting", season)
    
    adv_args = {
        "season": season,
        "season_type_all_star": "Regular Season",
        "measure_type_detailed_defense": "Advanced",
        "per_mode_detailed": "Per100Possessions",
    }

    adv_df = cached_api_call(LeagueDashPlayerStats, "LeagueDashPlayerStats", adv_args)

    adv_oreb_cols = ["PLAYER_ID", "OREB_PCT"]

    adv_oreb_df = adv_df[adv_oreb_cols]

    # Rebound Tracking data
    tracking_args = {
        "season": season,
        "season_type_all_star": "Regular Season",
        "pt_measure_type": "Rebounding",
        "per_mode_simple": "Totals",
        "player_or_team": "Player"
    }

    tracking_reb_df = cached_api_call(LeagueDashPtStats, "LeagueDashPtStats", tracking_args)

    tracking_reb_df["OREB_DEFER_PCT"] = tracking_reb_df["OREB_CHANCE_DEFER"] / tracking_reb_df["OREB_CHANCES"]
    tracking_reb_df["DREB_DEFER_PCT"] = tracking_reb_df["DREB_CHANCE_DEFER"] / tracking_reb_df["DREB_CHANCES"]

    tracking_oreb_cols = [
        "PLAYER_ID",
        "OREB_CONTEST_PCT",
        "OREB_CHANCE_PCT",
        "AVG_OREB_DIST",
        "OREB_DEFER_PCT",
    ]

    tracking_oreb_df = tracking_reb_df[tracking_oreb_cols]

    # Hustle data
    hustle_args = {
        "season": season,
        "season_type_all_star": "Regular Season",
        "per_mode_time": "Totals",
    }

    hustle_df = cached_api_call(LeagueHustleStatsPlayer, "LeagueHustleStatsPlayer", hustle_args)

    hustle_oreb_cols = [
        "PLAYER_ID",
        "PLAYER_NAME",
        "MIN",
        "PCT_BOX_OUTS_OFF",
        # Not offensive specific:
        "PCT_BOX_OUTS_TEAM_REB",
        "PCT_BOX_OUTS_REB"
    ]

    hustle_oreb_df = hustle_df[hustle_oreb_cols]

    usage_args = {
        "season": "2025-26",
        "season_type_all_star": "Regular Season",
        "measure_type_detailed_defense": "Usage",
        "per_mode_detailed": "Per100Possessions",
    }

    usage_df = cached_api_call(LeagueDashPlayerStats, "LeagueDashPlayerStats", usage_args)

    # Renaming unclear column name
    usage_df = usage_df.rename(columns={"PCT_OREB": "PCT_TEAM_OREB"})

    usage_oreb_cols = [
        "PLAYER_ID",
        "PCT_TEAM_OREB"
    ]

    usage_oreb_df = usage_df[usage_oreb_cols]

    oreb_target_df = targets_dict[season]

    oreb_target_df = oreb_target_df[[
            'PLAYER_ID',
            'TM_OREB_PCT',
        ]]

    # Merge dataframes
    oreb_dfs = [adv_oreb_df, tracking_oreb_df, hustle_oreb_df, usage_oreb_df, oreb_target_df]
    oreb_df = reduce(lambda left, right: pd.merge(left, right, on='PLAYER_ID', how='outer'), oreb_dfs)

    oreb_df_dict[season] = oreb_df
    time.sleep(2)



Starting 2021-22
Loading cached: cache/LeagueDashPlayerStats_2021-22_Regular_Season_Advanced_Per100Possessions.csv
Loading cached: cache/LeagueDashPtStats_2021-22_Regular_Season_Rebounding_Totals_Player.csv
Loading cached: cache/LeagueHustleStatsPlayer_2021-22_Regular_Season_Totals.csv
Loading cached: cache/LeagueDashPlayerStats_2025-26_Regular_Season_Usage_Per100Possessions.csv
Starting 2022-23
Loading cached: cache/LeagueDashPlayerStats_2022-23_Regular_Season_Advanced_Per100Possessions.csv
Loading cached: cache/LeagueDashPtStats_2022-23_Regular_Season_Rebounding_Totals_Player.csv
Loading cached: cache/LeagueHustleStatsPlayer_2022-23_Regular_Season_Totals.csv
Loading cached: cache/LeagueDashPlayerStats_2025-26_Regular_Season_Usage_Per100Possessions.csv
Starting 2023-24
Loading cached: cache/LeagueDashPlayerStats_2023-24_Regular_Season_Advanced_Per100Possessions.csv
Loading cached: cache/LeagueDashPtStats_2023-24_Regular_Season_Rebounding_Totals_Player.csv
Loading cached: cache/LeagueH

In [55]:
oreb_df_dict["2025-26"]

,PLAYER_ID,OREB_PCT,OREB_CONTEST_PCT,OREB_CHANCE_PCT,AVG_OREB_DIST,OREB_DEFER_PCT,PLAYER_NAME,MIN,PCT_BOX_OUTS_OFF,PCT_BOX_OUTS_TEAM_REB,PCT_BOX_OUTS_REB,PCT_TEAM_OREB,TM_OREB_PCT
0,2544,0.023,0.395,0.467,8.3,0.108696,LeBron James,1989.0,0.115,1.0,0.769,0.109,0.291
1,101108,0.026,0.333,0.462,10.8,0.153846,Chris Paul,207.0,0.000,1.0,0.400,0.111,0.286
2,200768,0.000,0.000,0.000,0.0,0.500000,Kyle Lowry,91.0,0.000,1.0,0.000,0.000,0.294
3,201142,0.014,0.244,0.594,16.7,0.086957,Kevin Durant,2840.0,0.000,1.0,0.600,0.048,0.382
4,201143,0.043,0.500,0.638,9.0,0.014493,Al Horford,969.0,0.105,1.0,0.895,0.216,0.274
...,...,...,...,...,...,...,...,...,...,...,...,...,...
577,1643133,0.105,1.000,0.750,3.9,0.125000,Lawson Lovering,49.0,0.000,1.0,0.667,0.400,0.316
578,1643141,0.000,0.000,0.000,0.0,0.000000,Jahmyl Telfort,11.0,0.000,1.0,0.000,0.000,0.267
579,1643158,0.089,0.765,0.773,6.2,0.181818,Andersson Garcia,169.0,0.200,1.0,0.600,0.333,0.340
580,1643253,0.034,0.444,0.450,8.6,0.150000,Toby Okani,217.0,0.000,1.0,0.333,0.188,0.232


In [56]:
df_26 = oreb_df_dict['2025-26']
df_26[df_26['PLAYER_NAME'] == 'Jaren Jackson Jr.']

,PLAYER_ID,OREB_PCT,OREB_CONTEST_PCT,OREB_CHANCE_PCT,AVG_OREB_DIST,OREB_DEFER_PCT,PLAYER_NAME,MIN,PCT_BOX_OUTS_OFF,PCT_BOX_OUTS_TEAM_REB,PCT_BOX_OUTS_REB,PCT_TEAM_OREB,TM_OREB_PCT
134,1628991,0.027,0.738,0.568,7.2,0.094595,Jaren Jackson Jr.,1455.0,0.079,0.974,0.579,0.126,0.311495


In [162]:
oreb_features = [ 
    'OREB_PCT', 
    'OREB_CONTEST_PCT', 
    'OREB_CHANCE_PCT',
    'AVG_OREB_DIST', 
    'OREB_DEFER_PCT', 
    'PCT_BOX_OUTS_OFF', 
    # 'PCT_BOX_OUTS_TEAM_REB', # unused because they include defense
    # 'PCT_BOX_OUTS_REB',
    'PCT_TEAM_OREB'
]

In [58]:
df_26 = oreb_df_dict['2025-26']
len(df_26[df_26['PLAYER_NAME'] == 'Jaren Jackson Jr.'])

1

In [60]:
# Filter players with less than 500 min
for season in oreb_df_dict.keys():
    df = oreb_df_dict[season]
    oreb_df_dict[season] = df[df["MIN"] > 500]
    

In [61]:
# oreb_df_dict["2025-26"]["OREB_PCT"].max()
oreb_df_dict["2025-26"].columns

Index(['PLAYER_ID', 'OREB_PCT', 'OREB_CONTEST_PCT', 'OREB_CHANCE_PCT',
       'AVG_OREB_DIST', 'OREB_DEFER_PCT', 'PLAYER_NAME', 'MIN',
       'PCT_BOX_OUTS_OFF', 'PCT_BOX_OUTS_TEAM_REB', 'PCT_BOX_OUTS_REB',
       'PCT_TEAM_OREB', 'TM_OREB_PCT'],
      dtype='object')

Target = on court team OREB% diff - league avg team OREB%

adjust features to difference from avg in each season. standard scaler does this to some extent, but we want to make sure the season by season variation is accounted for

In [62]:
oreb_target = ['TM_OREB_PCT']

In [63]:
oreb_df_dict["2025-26"].iloc[0]

PLAYER_ID                        2544
OREB_PCT                        0.023
OREB_CONTEST_PCT                0.395
OREB_CHANCE_PCT                 0.467
AVG_OREB_DIST                     8.3
OREB_DEFER_PCT               0.108696
PLAYER_NAME              LeBron James
MIN                            1989.0
PCT_BOX_OUTS_OFF                0.115
PCT_BOX_OUTS_TEAM_REB             1.0
PCT_BOX_OUTS_REB                0.769
PCT_TEAM_OREB                   0.109
TM_OREB_PCT                     0.291
Name: 0, dtype: object

In [ ]:
# Adjusting each value relative to mean for season
for season, df in oreb_df_dict.items():
    df_clean = df.copy()
    
    df_clean[oreb_features] = df_clean[oreb_features] - df_clean[oreb_features].mean()
    df_clean[oreb_target] = df_clean[oreb_target] - df_clean[oreb_target].mean()
    
    oreb_df_dict[season] = df_clean



In [65]:
for season, df in oreb_df_dict.items():
    df_clean = df.copy()
    
    df_clean["SEASON"] = season
    
    oreb_df_dict[season] = df_clean

In [ ]:
season_order = {
    "2021-22":4,
    "2022-23":3,
    "2023-24":2,
    "2024-25":1,
    "2025-26":0
}

for season, df in oreb_df_dict.items():
    df_clean = df.copy()
    
    df_clean["YEARS_SINCE"] = (
            df_clean["SEASON"]
            .map(season_order)
        )
    
    oreb_df_dict[season] = df_clean

In [155]:
oreb_df_dict["2021-22"].iloc[0]

PLAYER_ID                        2544
OREB_PCT                    -0.011866
OREB_CONTEST_PCT             0.058306
OREB_CHANCE_PCT             -0.027249
AVG_OREB_DIST               -3.553887
OREB_DEFER_PCT              -0.023403
PLAYER_NAME              LeBron James
MIN                            2084.0
PCT_BOX_OUTS_OFF             0.075547
PCT_BOX_OUTS_TEAM_REB        0.050107
PCT_BOX_OUTS_REB             0.316777
PCT_TEAM_OREB               -0.091574
TM_OREB_PCT                 -0.033372
SEASON                        2021-22
YEARS_SINCE                         4
Name: 1, dtype: object

In [109]:
oreb_df_dict["2025-26"].iloc[0]

PLAYER_ID                        2544
OREB_PCT                    -0.023557
OREB_CONTEST_PCT            -0.087496
OREB_CHANCE_PCT              -0.11352
AVG_OREB_DIST               -1.417067
OREB_DEFER_PCT               0.006833
PLAYER_NAME              LeBron James
MIN                            1989.0
PCT_BOX_OUTS_OFF            -0.050821
PCT_BOX_OUTS_TEAM_REB        0.011275
PCT_BOX_OUTS_REB             0.235304
PCT_TEAM_OREB               -0.092021
TM_OREB_PCT                 -0.014687
SEASON                        2025-26
YEARS_SINCE                         0
Name: 0, dtype: object

In [163]:
oreb_train_df = pd.DataFrame()

for season, df in oreb_df_dict.items():
    if season == "2025-26":
        break
    df_clean = df.copy()
    
    oreb_train_df = pd.concat([oreb_train_df, df_clean], axis=0)

In [164]:
oreb_test_df = oreb_df_dict["2025-26"].copy()

In [165]:
X_train_oreb = oreb_train_df.drop(oreb_target, axis=1)
y_train_oreb = oreb_train_df[oreb_target]

X_test_oreb = oreb_test_df.drop(oreb_target, axis=1)
y_test_oreb = oreb_test_df[oreb_target]

In [256]:
oreb_drop_features = [
    "PLAYER_ID",
    "PLAYER_NAME",
    "MIN",
    "SEASON",
    'PCT_BOX_OUTS_TEAM_REB', 
    'PCT_BOX_OUTS_REB',
    # 'PCT_TEAM_OREB'
]

In [ ]:
k = X_train_oreb["YEARS_SINCE"] 

oreb_param_grid = {
    'ridge__alpha': [
        .01,
        .1,
        1,
        5,
        10,
        50,
        100,
    ],
    'columntransformer__pipeline__iterativeimputer__max_iter': [
        5,
        10,
        20,
        50
    ],
    'columntransformer__pipeline__iterativeimputer__tol': [1e-3, 1e-4]
}

# Weighting stats by season: recent seasons weighted more heavily
# Weighting: each stat multiplied by D^k, where:
# k is the # of years since most recent season, and 
# D is a constant we tune

d_oreb = 0.5
k_oreb = X_train_oreb["YEARS_SINCE"] 
season_weight_dreb = d_oreb ** k_oreb

# Fills in missing numbers with the mean for that column, 
# and scales the numbers to normal distribution
# oreb_numeric_transformer = make_pipeline(SimpleImputer(strategy="mean"), StandardScaler())
oreb_numeric_transformer = make_pipeline(IterativeImputer(random_state=42), StandardScaler())

oreb_preprocessor = make_column_transformer(
    (oreb_numeric_transformer, oreb_features),   
    ("drop", oreb_drop_features), 
    verbose_feature_names_out = False
)

oreb_model = make_pipeline(oreb_preprocessor, Ridge())

oreb_grid = GridSearchCV(estimator=oreb_model, param_grid=oreb_param_grid, cv=5, scoring="neg_root_mean_squared_error", n_jobs=-1)

oreb_grid.fit(X_train_oreb, y_train_oreb, ridge__sample_weight=season_weight_oreb)

print("Best Parameters Found: ", oreb_grid.best_params_)
print("Best Cross-Validation Score: ", oreb_grid.best_score_)

Best Parameters Found:  {'columntransformer__pipeline__iterativeimputer__max_iter': 5, 'columntransformer__pipeline__iterativeimputer__tol': 0.001, 'ridge__alpha': 5}
Best Cross-Validation Score:  -0.02773375603241862


In [180]:
oreb_model = make_pipeline(preprocessor, Ridge(alpha=5))

In [ ]:
oreb_model.fit(X=X_train_oreb, y=y_train_oreb, ridge__sample_weight=season_weight_ore)

,steps,"[('columntransformer', ...), ('ridge', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('pipeline', ...), ('drop', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,False


In [189]:
oreb_predictions = oreb_grid.predict(X_test_oreb)
oreb_results_df = oreb_test_df.assign(PREDICTED_TM_OREB_PCT=oreb_predictions)

In [ ]:
rmse_oreb = root_mean_squared_error(y_test_oreb, oreb_predictions)
rmse_oreb

0.03037303976467174

In [ ]:
mae_oreb = mean_absolute_error(y_test_oreb, oreb_predictions)
mae_oreb

0.023994866694181254

In [195]:
oreb_df_sorted = oreb_results_df.sort_values(by='PREDICTED_TM_OREB_PCT', ascending=False)
oreb_df_sorted[['PLAYER_NAME', 'PREDICTED_TM_OREB_PCT', 'TM_OREB_PCT']]

,PLAYER_NAME,PREDICTED_TM_OREB_PCT,TM_OREB_PCT
39,Steven Adams,0.082627,0.145313
141,Mitchell Robinson,0.081555,0.089313
57,Clint Capela,0.055169,0.086313
456,Donovan Clingan,0.052240,0.073313
29,Andre Drummond,0.048491,0.034313
...,...,...,...
177,Tyler Herro,-0.024693,-0.041687
209,Tyrese Maxey,-0.025958,0.003313
313,Max Christie,-0.026240,-0.038687
10,Brook Lopez,-0.027572,-0.063687


In [196]:
oreb_df_sorted[['PLAYER_NAME', 'PREDICTED_TM_OREB_PCT']].to_csv("outputs/multi_layer/oreb/oreb_results_iterativeimputer.csv", index=False)

In [197]:
oreb_df_sorted[['PLAYER_NAME', 'TM_OREB_PCT', 'PREDICTED_TM_OREB_PCT']].to_csv("outputs/multi_layer/oreb/oreb_results_w_original_iterativeimputer.csv", index=False)

In [209]:
oreb_df_sorted[['PLAYER_NAME']].head(10).to_csv("outputs/multi_layer/oreb/top_10_names_oreb_results_v1.csv", index=False)

In [204]:
oreb_coefs = pd.DataFrame(
    data=oreb_grid.best_estimator_.named_steps["ridge"].coef_, index=oreb_features, columns=["Coefficient"]
)

In [205]:
oreb_coefs.sort_values(by="Coefficient", ascending=False).head(80)

,Coefficient
OREB_PCT,0.023403
PCT_BOX_OUTS_OFF,0.004942
OREB_CHANCE_PCT,0.002248
AVG_OREB_DIST,0.002030
OREB_DEFER_PCT,0.001872
OREB_CONTEST_PCT,-0.005504
PCT_TEAM_OREB,-0.010765


# DREB model

In [208]:
adv_dreb_cols = ["PLAYER_ID", "DREB_PCT"]

In [209]:
tracking_dreb_cols = [
    "PLAYER_ID",
    "DREB_CONTEST_PCT",
    "DREB_CHANCE_PCT",
    "AVG_DREB_DIST",
    "DREB_DEFER_PCT",
]

In [219]:
hustle_dreb_cols = [
    "PLAYER_ID",
    "PLAYER_NAME",
    "MIN",
    "PCT_BOX_OUTS_DEF",
    # Not defensive specific:
    "PCT_BOX_OUTS_TEAM_REB",
    "PCT_BOX_OUTS_REB"
]

In [242]:
dreb_df_dict = {}

for season in seasons:

    print("Starting", season)
    
    adv_args = {
        "season": season,
        "season_type_all_star": "Regular Season",
        "measure_type_detailed_defense": "Advanced",
        "per_mode_detailed": "Per100Possessions",
    }

    adv_df = cached_api_call(LeagueDashPlayerStats, "LeagueDashPlayerStats", adv_args)

    adv_dreb_df = adv_df[adv_dreb_cols]

    # Rebound Tracking data
    tracking_args = {
        "season": season,
        "season_type_all_star": "Regular Season",
        "pt_measure_type": "Rebounding",
        "per_mode_simple": "Totals",
        "player_or_team": "Player"
    }

    tracking_reb_df = cached_api_call(LeagueDashPtStats, "LeagueDashPtStats", tracking_args)

    # tracking_reb_df["OREB_DEFER_PCT"] = tracking_reb_df["OREB_CHANCE_DEFER"] / tracking_reb_df["OREB_CHANCES"]
    tracking_reb_df["DREB_DEFER_PCT"] = tracking_reb_df["DREB_CHANCE_DEFER"] / tracking_reb_df["DREB_CHANCES"]

    tracking_dreb_df = tracking_reb_df[tracking_dreb_cols]

    # Hustle data
    hustle_args = {
        "season": season,
        "season_type_all_star": "Regular Season",
        "per_mode_time": "Totals",
    }

    hustle_df = cached_api_call(LeagueHustleStatsPlayer, "LeagueHustleStatsPlayer", hustle_args)

    hustle_dreb_df = hustle_df[hustle_dreb_cols]

    usage_args = {
        "season": "2025-26",
        "season_type_all_star": "Regular Season",
        "measure_type_detailed_defense": "Usage",
        "per_mode_detailed": "Per100Possessions",
    }

    usage_df = cached_api_call(LeagueDashPlayerStats, "LeagueDashPlayerStats", usage_args)

    # Renaming unclear column name
    usage_df = usage_df.rename(columns={"PCT_DREB": "PCT_TEAM_DREB"})

    usage_dreb_cols = [
        "PLAYER_ID",
        "PCT_TEAM_DREB"
    ]

    usage_dreb_df = usage_df[usage_dreb_cols]

    dreb_target_df = targets_dict[season]

    dreb_target_df = dreb_target_df[[
            'PLAYER_ID',
            'OPP_OREB_PCT',
        ]]

    # Merge dataframes
    dreb_dfs = [adv_dreb_df, tracking_dreb_df, hustle_dreb_df, usage_dreb_df, dreb_target_df]
    dreb_df = reduce(lambda left, right: pd.merge(left, right, on='PLAYER_ID', how='outer'), dreb_dfs)

    dreb_df_dict[season] = dreb_df
    time.sleep(2)

Starting 2021-22
Loading cached: cache/LeagueDashPlayerStats_2021-22_Regular_Season_Advanced_Per100Possessions.csv
Loading cached: cache/LeagueDashPtStats_2021-22_Regular_Season_Rebounding_Totals_Player.csv
Loading cached: cache/LeagueHustleStatsPlayer_2021-22_Regular_Season_Totals.csv
Loading cached: cache/LeagueDashPlayerStats_2025-26_Regular_Season_Usage_Per100Possessions.csv
Starting 2022-23
Loading cached: cache/LeagueDashPlayerStats_2022-23_Regular_Season_Advanced_Per100Possessions.csv
Loading cached: cache/LeagueDashPtStats_2022-23_Regular_Season_Rebounding_Totals_Player.csv
Loading cached: cache/LeagueHustleStatsPlayer_2022-23_Regular_Season_Totals.csv
Loading cached: cache/LeagueDashPlayerStats_2025-26_Regular_Season_Usage_Per100Possessions.csv
Starting 2023-24
Loading cached: cache/LeagueDashPlayerStats_2023-24_Regular_Season_Advanced_Per100Possessions.csv
Loading cached: cache/LeagueDashPtStats_2023-24_Regular_Season_Rebounding_Totals_Player.csv
Loading cached: cache/LeagueH

In [246]:
dreb_features = [ 
    'DREB_PCT', 
    'DREB_CONTEST_PCT', 
    'DREB_CHANCE_PCT',
    'AVG_DREB_DIST', 
    'DREB_DEFER_PCT', 
    'PCT_BOX_OUTS_DEF', 
    # 'PCT_BOX_OUTS_TEAM_REB', # unused because they include defense
    # 'PCT_BOX_OUTS_REB',
    'PCT_TEAM_DREB'
]

In [244]:
# Filter players with less than 500 min
for season in dreb_df_dict.keys():
    df = dreb_df_dict[season]
    dreb_df_dict[season] = df[df["MIN"] > 500]
    

In [247]:
dreb_target = ['OPP_OREB_PCT']

In [248]:
dreb_df_dict["2025-26"].iloc[0]

PLAYER_ID                        2544
DREB_PCT                         0.16
DREB_CONTEST_PCT                0.221
DREB_CHANCE_PCT                 0.823
AVG_DREB_DIST                     6.7
DREB_DEFER_PCT               0.169231
PLAYER_NAME              LeBron James
MIN                            1989.0
PCT_BOX_OUTS_DEF                0.885
PCT_BOX_OUTS_TEAM_REB             1.0
PCT_BOX_OUTS_REB                0.769
PCT_TEAM_DREB                   0.246
OPP_OREB_PCT                    0.304
Name: 0, dtype: object

In [249]:
# Adjusting each value relative to mean for season
for season, df in dreb_df_dict.items():
    df_clean = df.copy()
    
    df_clean[dreb_features] = df_clean[dreb_features] - df_clean[dreb_features].mean()
    df_clean[dreb_target] = df_clean[dreb_target] - df_clean[dreb_target].mean()
    
    dreb_df_dict[season] = df_clean

In [250]:
for season, df in dreb_df_dict.items():
    df_clean = df.copy()
    
    df_clean["SEASON"] = season
    
    dreb_df_dict[season] = df_clean

In [251]:
season_order = {
    "2021-22":4,
    "2022-23":3,
    "2023-24":2,
    "2024-25":1,
    "2025-26":0
}

for season, df in dreb_df_dict.items():
    df_clean = df.copy()
    
    df_clean["YEARS_SINCE"] = (
            df_clean["SEASON"]
            .map(season_order)
        )
    
    dreb_df_dict[season] = df_clean

In [252]:
dreb_df_dict["2025-26"].iloc[0]

PLAYER_ID                        2544
DREB_PCT                     0.030851
DREB_CONTEST_PCT            -0.020608
DREB_CHANCE_PCT              0.092885
AVG_DREB_DIST               -0.405067
DREB_DEFER_PCT              -0.013257
PLAYER_NAME              LeBron James
MIN                            1989.0
PCT_BOX_OUTS_DEF             0.051349
PCT_BOX_OUTS_TEAM_REB             1.0
PCT_BOX_OUTS_REB                0.769
PCT_TEAM_DREB                0.046245
OPP_OREB_PCT                -0.002913
SEASON                        2025-26
YEARS_SINCE                         0
Name: 0, dtype: object

In [253]:
dreb_train_df = pd.DataFrame()

for season, df in dreb_df_dict.items():
    if season == "2025-26":
        break
    df_clean = df.copy()
    
    dreb_train_df = pd.concat([dreb_train_df, df_clean], axis=0)

In [254]:
dreb_test_df = dreb_df_dict["2025-26"].copy()

In [262]:
X_train_dreb = dreb_train_df.drop(dreb_target, axis=1)
y_train_dreb = dreb_train_df[dreb_target]

X_test_dreb = dreb_test_df.drop(dreb_target, axis=1)
y_test_dreb = dreb_test_df[dreb_target]

In [263]:
dreb_drop_features = [
    "PLAYER_ID",
    "PLAYER_NAME",
    "MIN",
    "SEASON",
    'PCT_BOX_OUTS_TEAM_REB', 
    'PCT_BOX_OUTS_REB',
    # 'PCT_TEAM_OREB'
]

In [280]:
k = X_train_dreb["YEARS_SINCE"] 

dreb_param_grid = {
    'ridge__alpha': [
        .01,
        .1,
        1,
        5,
        10,
        50,
        100,
    ],
    'columntransformer__pipeline__iterativeimputer__max_iter': [
        5,
        10,
        20,
        50
    ],
    'columntransformer__pipeline__iterativeimputer__tol': [1e-3, 1e-4]
}

# Weighting stats by season: recent seasons weighted more heavily
# Weighting: each stat multiplied by D^k, where:
# k is the # of years since most recent season, and 
# D is a constant we tune

d_dreb = 0.5
k_dreb = X_train_dreb["YEARS_SINCE"] 
season_weight_dreb = d_dreb ** k_dreb

# Fills in missing numbers with the mean for that column, 
# and scales the numbers to normal distribution
# dreb_numeric_transformer = make_pipeline(SimpleImputer(strategy="mean"), StandardScaler())
dreb_numeric_transformer = make_pipeline(IterativeImputer(random_state=42), StandardScaler())

dreb_preprocessor = make_column_transformer(
    (dreb_numeric_transformer, dreb_features),   
    ("drop", dreb_drop_features), 
    verbose_feature_names_out = False
)

dreb_model = make_pipeline(dreb_preprocessor, Ridge())

dreb_grid = GridSearchCV(estimator=dreb_model, param_grid=dreb_param_grid, cv=5, scoring="neg_root_mean_squared_error", n_jobs=-1)

dreb_grid.fit(X_train_dreb, y_train_dreb, ridge__sample_weight=season_weight_dreb)

print("Best Parameters Found: ", dreb_grid.best_params_)
print("Best Cross-Validation Score: ", dreb_grid.best_score_)

Best Parameters Found:  {'columntransformer__pipeline__iterativeimputer__max_iter': 5, 'columntransformer__pipeline__iterativeimputer__tol': 0.001, 'ridge__alpha': 10}
Best Cross-Validation Score:  -0.019678628057413095


In [265]:
dreb_predictions = dreb_grid.predict(X_test_dreb)
dreb_results_df = dreb_test_df.assign(PREDICTED_OPP_OREB_PCT=dreb_predictions)

In [266]:
rmse_dreb = root_mean_squared_error(y_test_dreb, dreb_predictions)
rmse_dreb

0.021609483174571332

In [268]:
mae_dreb = mean_absolute_error(y_test_dreb, dreb_predictions)
mae_dreb

0.017141589847153

In [269]:
dreb_df_sorted = dreb_results_df.sort_values(by='PREDICTED_OPP_OREB_PCT', ascending=True)
dreb_df_sorted[['PLAYER_NAME', 'PREDICTED_OPP_OREB_PCT', 'OPP_OREB_PCT']]

,PLAYER_NAME,PREDICTED_OPP_OREB_PCT,OPP_OREB_PCT
59,Jusuf Nurkić,-0.023337,-0.066913
554,Dylan Cardwell,-0.020820,0.011087
154,Robert Williams III,-0.020664,0.019087
66,Karl-Anthony Towns,-0.020495,-0.036913
213,Jalen Smith,-0.019286,-0.035913
...,...,...,...
128,Jevon Carter,0.015974,0.009876
244,Bones Hyland,0.015974,0.008087
44,Zach LaVine,0.015988,0.016087
335,Keon Ellis,0.018902,0.029908


In [270]:
dreb_coefs = pd.DataFrame(
    data=dreb_grid.best_estimator_.named_steps["ridge"].coef_, index=dreb_features, columns=["Coefficient"]
)

In [271]:
dreb_coefs.sort_values(by="Coefficient", ascending=True).head(80)

,Coefficient
DREB_CHANCE_PCT,-0.004713
DREB_PCT,-0.004701
DREB_DEFER_PCT,-0.003768
AVG_DREB_DIST,-0.002910
DREB_CONTEST_PCT,-0.001899
PCT_BOX_OUTS_DEF,-0.000505
PCT_TEAM_DREB,-0.000448


In [273]:
dreb_df_sorted[['PLAYER_NAME', 'PREDICTED_OPP_OREB_PCT']].to_csv("outputs/multi_layer/dreb/dreb_results_iterativeimputer.csv", index=False)

In [274]:
dreb_df_sorted[['PLAYER_NAME', 'OPP_OREB_PCT', 'PREDICTED_OPP_OREB_PCT']].to_csv("outputs/multi_layer/dreb/dreb_results_w_original_iterativeimputer.csv", index=False)

In [282]:
dreb_df_sorted[['PLAYER_NAME']].tail(10).to_csv("outputs/multi_layer/dreb/bottom_10_names_dreb_results_v1.csv", index=False)